# Experiment: BrandGraph Event Flow

Objective:
- Generate a sample tenant + brand + artifact link payload
- Simulate a rotation handoff snapshot payload
- Validate payloads against a minimal schema contract

Done means:
- All assertions pass
- Printed JSON artifacts are stable and deterministic


In [ ]:
from __future__ import annotations

import json
import hashlib
from datetime import datetime, timezone

now = datetime(2026, 2, 6, 0, 0, 0, tzinfo=timezone.utc)

TENANT_ID = "tenant_demo"
BRAND_ID = "brand_demo"
ARTIFACT_ID = "art_demo"

AGENT_ID = "brandyn-v1"
AGENT_VERSION = "v1"
POLICY_ID = "PROMPT_CONSTITUTION_v1"


## Deterministic helpers


In [ ]:
def deterministic_id(*parts: str) -> str:
    raw = ":".join(parts).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:24]


def iso(ts: datetime) -> str:
    return ts.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


## Build payloads


In [ ]:
brand_created_event = {
    "id": deterministic_id(TENANT_ID, BRAND_ID, "BRAND_CREATED"),
    "tenantId": TENANT_ID,
    "brandId": BRAND_ID,
    "eventType": "BRAND_CREATED",
    "payload": {
        "brandName": "Demo Brand",
        "createdAt": iso(now)
    },
    "createdAt": iso(now)
}

artifact_linked_event = {
    "id": deterministic_id(TENANT_ID, BRAND_ID, ARTIFACT_ID, "ARTIFACT_LINKED"),
    "tenantId": TENANT_ID,
    "brandId": BRAND_ID,
    "eventType": "ARTIFACT_LINKED",
    "payload": {
        "artifactId": ARTIFACT_ID,
        "artifactType": "BrandBrief"
    },
    "createdAt": iso(now)
}

handoff_snapshot = {
    "fromAgentId": AGENT_ID,
    "toAgentId": "brandyn-v2",
    "tenantId": TENANT_ID,
    "memoryNamespace": "brand-trinity/brandyn",
    "createdAt": iso(now),
    "contextRefs": [
        {"type": "agent-manifest", "ref": AGENT_ID},
        {"type": "rotation-reason", "ref": "EXPIRING_SOON"}
    ],
    "payload": {
        "rotatedAt": iso(now),
        "brandId": BRAND_ID,
        "artifactId": ARTIFACT_ID
    }
}


## Minimal schema checks


In [ ]:
def require_keys(obj, keys):
    missing = [k for k in keys if k not in obj]
    if missing:
        raise AssertionError(f"Missing keys: {missing}")

require_keys(brand_created_event, ["id", "tenantId", "brandId", "eventType", "payload", "createdAt"])
require_keys(artifact_linked_event, ["id", "tenantId", "brandId", "eventType", "payload", "createdAt"])
require_keys(handoff_snapshot, ["fromAgentId", "toAgentId", "tenantId", "memoryNamespace", "createdAt", "contextRefs", "payload"])

print(json.dumps(brand_created_event, indent=2))
print(json.dumps(artifact_linked_event, indent=2))
print(json.dumps(handoff_snapshot, indent=2))
